# The Application layer

As every person who have used the internet, you are probably 
aware that we interact with remote devices via the internet using applications.

These remote devices may be from our family and friends. Or they can be from 
complete strangers that rent their servers for another strangers that want
a place to host their own applications. These infrastructure providers are
currently called cloud service providers (e.g. AWS, Azure, etc), and the 
renters are known as tenant customers.

Let's do a very simple network topology.

```mermaid
flowchart
direction LR
A[Node A]-->|P2P link: 5 Mbps, 2ms delay|B[Node B]
```

Now that we have decided on our network topology, we can implement it in ns-3.

In [1]:
import sys
sys.path.append("./ns-3-dev/build/bindings/python")
sys.path.append("./ns-3-dev/build/lib")
from ns import ns

nodes = ns.NodeContainer()
nodes.Create(2)

pointToPoint = ns.PointToPointHelper()
pointToPoint.SetDeviceAttribute("DataRate", ns.StringValue("5Mbps"))
pointToPoint.SetChannelAttribute("Delay", ns.StringValue("2ms"))

devices = pointToPoint.Install(nodes)
pointToPoint.EnablePcapAll("./application", True)

stack = ns.InternetStackHelper()
stack.Install(nodes)

address = ns.Ipv4AddressHelper()
address.SetBase(ns.Ipv4Address("10.1.1.0"),
                ns.Ipv4Mask("255.255.255.0"))

interfaces = address.Assign(devices)

In the setup above we have two devices, called Node 0 and Node 1.

These network nodes, as they are usually called, can contain multiple network interfaces. 

Think of your phone: it has 4G, 5G, Wi-Fi, Bluetooth, NFC, plus the USB port used for charging and data. 

Some may have additional satellite communication capabilities, others walk-talkie like capabilities, maybe even a infrared interface, or a lightning port for Apple afficionados.

All of these network interfaces are also called network devices (or NetDevice in ns-3). 

Each of them will contain at least the Medium Access Control (MAC) and Physical Layers (PHY), which are used to exchange messages via a link, be it based on radiofrequency, acoustics, light, etc. 

We are not going to focus on network devices now. But focus on the top.

Notice we have an IP layer. IP stands for Internet Procotol, which is the most commonly used network layer protocol because it is the one used to communicate in the internet. Operating systems or standalone network stacks can use many other network layer protocols, such as [those available on Linux](https://github.com/torvalds/linux/blob/master/net/socket.c#L170-L217).

The IP layer is responsible for routing our application packages to their destination.

But we currently don't have any application, so let's add one.

[![](https://mermaid.ink/img/pako:eNqVk1FrgzAUhf9KyLMO47aXsBZcV6h0FZntw9A9pOZ2lWmUGAej9L8vadywa0uZQtR7DudeP80O5zUHTPG7ZM0WLR8zgfTRdmtbiLTqodRckPdmxSPDKtHyqgWJ2oblMLAEceylK95M8209KQsQiiLt-wQZNiPi3RB9-k5fimupRsS_vRsEgOBnGs5NwzlIAeVJy-VLECW2K0WSCV5XqNHJA0uo5TCmqB-ADKRFMPFSvQxK8ezVS_VyaSrzjg-um9T5ByjXHfcDWMPBeYYn6XmS8zzJZZ7kh2dygEZRWbQKxH_gkSvwiIVnqKE_kaEWB-j8Y3TkFB25go6coiO_6KzLPk5mQRRNn9N7tFg3rYN8VLV9rvlCOmbcW478pm7GwA6uQFas4PpH3xlLhtUWKsgw1bccNqwrVYYzsddW1qk6-RI5pkp24OCu4UzBU8E0wgrTDStbXQVeqFou7OY57KH9N3Ti7rk?type=png)](https://mermaid.live/edit#pako:eNqVk1FrgzAUhf9KyLMO47aXsBZcV6h0FZntw9A9pOZ2lWmUGAej9L8vadywa0uZQtR7DudeP80O5zUHTPG7ZM0WLR8zgfTRdmtbiLTqodRckPdmxSPDKtHyqgWJ2oblMLAEceylK95M8209KQsQiiLt-wQZNiPi3RB9-k5fimupRsS_vRsEgOBnGs5NwzlIAeVJy-VLECW2K0WSCV5XqNHJA0uo5TCmqB-ADKRFMPFSvQxK8ezVS_VyaSrzjg-um9T5ByjXHfcDWMPBeYYn6XmS8zzJZZ7kh2dygEZRWbQKxH_gkSvwiIVnqKE_kaEWB-j8Y3TkFB25go6coiO_6KzLPk5mQRRNn9N7tFg3rYN8VLV9rvlCOmbcW478pm7GwA6uQFas4PpH3xlLhtUWKsgw1bccNqwrVYYzsddW1qk6-RI5pkp24OCu4UzBU8E0wgrTDStbXQVeqFou7OY57KH9N3Ti7rk)

Now that we have an application, we can see that we had two layers missing. The top two ones: application and transport layers.

The transport layer implements application multiplexing, which allows the same computer to run many services addressed by a port. And with a source port, the two applications can talk with each other on different nodes.

That transport layer can be accessed by our applications via the operating system application programming interface (API). Currently, the most common API format is known as [BSD Sockets](https://docs.freebsd.org/en/books/developers-handbook/sockets/), that were then ported to pretty much every operating system.

Finally, we have the application layer, where we have our applications. They can be anything connected to the internet, including but not limited to: internet banking, streaming, games, websites, etc.

### Applications in ns-3

Applications in ns-3 are slightly different than real applications since we have a simulator instead of an operating system. Let's take a look at the following diagram.

[![](https://mermaid.ink/img/pako:eNqdkz1vAjEMhv-KlZlO7XQDCywMVVFP3W5xE0MjcknqOFSI8t_r61GVgjrATb58PO9rO94bmxyZxhR6rxQtzT2uGfsugn7PhAEw5-Atik_xbjp9ysQaxzWUXRHqP1vf14CSuNHjyihSYLHcPsDLfAkl2Q3JCPv_pmLPlQaYJb-lcmRAYiDmxCMslvszY2crDcyYUAgwXjjBIMAj3p1Cb0pZ2KtJdSfEvY-DpLzRKWKEUyj0q3pq5wbZlqIrkPEv5LIo1zOu7NIiatpc8_AHH7jRUgzZZ06WisbpJ2XA66Qu2tmOz-DZbmcYwquywBewGpM7VjhqYCam10agd_qo98NGZ9RRT51pNHS0whqkM1086FGsktpdtKYRrjQxNTtt4HEGTLNCbdrEkPNq6nEclO95OXwBCO8irg?type=png)](https://mermaid.live/edit#pako:eNqdkz1vAjEMhv-KlZlO7XQDCywMVVFP3W5xE0MjcknqOFSI8t_r61GVgjrATb58PO9rO94bmxyZxhR6rxQtzT2uGfsugn7PhAEw5-Atik_xbjp9ysQaxzWUXRHqP1vf14CSuNHjyihSYLHcPsDLfAkl2Q3JCPv_pmLPlQaYJb-lcmRAYiDmxCMslvszY2crDcyYUAgwXjjBIMAj3p1Cb0pZ2KtJdSfEvY-DpLzRKWKEUyj0q3pq5wbZlqIrkPEv5LIo1zOu7NIiatpc8_AHH7jRUgzZZ06WisbpJ2XA66Qu2tmOz-DZbmcYwquywBewGpM7VjhqYCam10agd_qo98NGZ9RRT51pNHS0whqkM1086FGsktpdtKYRrjQxNTtt4HEGTLNCbdrEkPNq6nEclO95OXwBCO8irg)

As shown, instead of requesting a socket, we just create one from the application.

The other obvious difference is that instead of the application being awoken where it stopped then continue processing, we instead call a function known as a callback to receive and decide what to do with that packet.

Since we already have our topology ready, let's go straight to the application setup. 

First we will need to define two wrappers using C++.

In [2]:
# We need to use this weird trick of defining c++ stuff from Python because
# template forwarding isn't supported by our cppjit bindings.

ns.cppjit.cppdef("""
    #ifndef AppCallbackAndSend
    #define AppCallbackAndSend
    namespace ns3
    {
        Callback<void,Ptr<Socket> > make_rx_callback(void(*func)(Ptr<Socket>))
        {
            return MakeCallback(func);
        }
        EventImpl* pythonMakeEventSend(void (*f)(Ptr<Socket>, Ptr<Packet>, Address&), Ptr<Socket> socket, Ptr<Packet> packet, Address address)
        {
            return MakeEvent(f, socket, packet, address);
        }
    }
    #endif
""")

True

Next, we can write our own application. In this case, it will be the UdpEchoServer.

[![](https://mermaid.ink/img/pako:eNp1UbtuwzAM_BWCc_oDGrL0sXWpVy-EeI6FypKrR4oiyL9XruAWTVFNEnl3PJ4ubKOCDWe8VQSLByenJMsYqB0f40pSCGJnyrAxaG882jnee4dQ7o7H7TEgnZEMDQiaaRX7itKh2Dmb2A-7E27YL7BwZ-wCJEEp2xlaPciVTFNM75LUhdM_St2UoacOpNQV9ZelP-531s387wX4wAvSIk5bUpetPHKZsWBk066KSaovI4_h2qBSSxw-gmVTUsWB66pS9mDZTOJzq0Jdiem5p__1CddPeIqH4Q?type=png)](https://mermaid.live/edit#pako:eNp1UbtuwzAM_BWCc_oDGrL0sXWpVy-EeI6FypKrR4oiyL9XruAWTVFNEnl3PJ4ubKOCDWe8VQSLByenJMsYqB0f40pSCGJnyrAxaG882jnee4dQ7o7H7TEgnZEMDQiaaRX7itKh2Dmb2A-7E27YL7BwZ-wCJEEp2xlaPciVTFNM75LUhdM_St2UoacOpNQV9ZelP-531s387wX4wAvSIk5bUpetPHKZsWBk066KSaovI4_h2qBSSxw-gmVTUsWB66pS9mDZTOJzq0Jdiem5p__1CddPeIqH4Q)

In [3]:
import sys
class EchoServer(ns.Application):
    LOGGING = True
    ECHO_PORT = 1234 # Application default listening port
    socketToInstanceDict = {}

    def __init__(self, node: ns.Node, port=ECHO_PORT):
        super().__init__() # Calls ns.Application constructor
        self.__python_owns__ = False  # Allow C++ to destroy this object when Simulator::Destroy gets called
        self.port = port # Server listening port
        
        # Create an UDP socket
        self.m_socket = ns.Socket.CreateSocket(node,
                                               ns.TypeId.LookupByName("ns3::UdpSocketFactory"))
        # Bind that socket to a specific port and IP addresses to listen
        self.m_socket.Bind(ns.InetSocketAddress(ns.Ipv4Address.GetAny(), self.port).ConvertTo())
        
        # Creates the callback to handle packet reception using a static function of the EchoServerClass
        # (This is a workaround for the application to work from Python)
        self.m_socket.SetRecvCallback(ns.make_rx_callback(EchoServer._Receive))
        
        # Registers the socket as a dictionary key and associate it to this application instance
        EchoServer.socketToInstanceDict[self.m_socket] = self

    def __del__(self):
        # Remove the instance entry on the class dictioanry when the object is destroyed
        del EchoServer.socketToInstanceDict[self.m_socket]

    def Send(self, packet: ns.Packet, address: ns.Address) -> None:
        # Function that sends the packet to a target address.
        self.m_socket.SendTo(packet, 0, address)
        if EchoServer.LOGGING:
            inetAddress = ns.InetSocketAddress.ConvertFrom(address)
            print("At time +{s}s server sent {b} bytes from {ip} port {port}"
                  .format(s=ns.Simulator.Now().GetSeconds(),
                          b=packet.__deref__().GetSize(),
                          ip=inetAddress.GetIpv4(),
                          port=inetAddress.GetPort()),
                  file=sys.stderr,
                  flush=True)

    def Receive(self):
        # Function that receives a packet and the address of the sender
        address = ns.Address()
        packet = self.m_socket.RecvFrom(address)
        if EchoServer.LOGGING:
            inetAddress = ns.InetSocketAddress.ConvertFrom(address)
            print("At time +{s}s server received {b} bytes from {ip} port {port}"
                  .format(s=ns.Simulator.Now().GetSeconds(),
                          b=packet.__deref__().GetSize(),
                          ip=inetAddress.GetIpv4(),
                          port=inetAddress.GetPort()),
                  file=sys.stderr,
                  flush=True)
        # Schedule the Send function to send the packet back to its sender after 1 second
        event = ns.pythonMakeEventSend(EchoServer._Send, self.m_socket, packet, address)
        ns.Simulator.Schedule(ns.Seconds(1), event)

    @staticmethod
    def _Send(socket: ns.Socket, packet: ns.Packet, address: ns.Address):
        # Static function that identifies the EchoServer instance that is
        # supposed to send a packet and calls it to send
        instance = EchoServer.socketToInstanceDict[socket]
        instance.Send(packet, address)
        pass

    @staticmethod
    def _Receive(socket: ns.Socket) -> None:
        # Static function that identifies the EchoServer instance that is
        # supposed to receive a packet and calls it to receive
        instance = EchoServer.socketToInstanceDict[socket]
        instance.Receive()
        pass

After the application is ready, we can use it in our topology.

In [4]:
echoServer = EchoServer(nodes.Get(1))
nodes.Get(1).AddApplication(echoServer)

serverApps = ns.ApplicationContainer()
serverApps.Add(echoServer)
serverApps.Start(ns.Seconds(1.0))
serverApps.Stop(ns.Seconds(10.0))

address = interfaces.GetAddress(1).ConvertTo()

ns.LogComponentEnable("UdpEchoClientApplication", ns.LOG_LEVEL_INFO)
echoClient = ns.UdpEchoClientHelper(address, EchoServer.ECHO_PORT)
echoClient.SetAttribute("MaxPackets", ns.UintegerValue(10))
echoClient.SetAttribute("Interval", ns.TimeValue(ns.Seconds(1.0)))
echoClient.SetAttribute("PacketSize", ns.UintegerValue(101))

clientApps = echoClient.Install(nodes.Get(0))
clientApps.Start(ns.Seconds(2.0))
clientApps.Stop(ns.Seconds(10.0))

ns.Simulator.Run()
ns.Simulator.Destroy()
ns.FatalImpl.FlushStreams() # Make sure the pcaps are written

Failed to GetVariableOffset:Symbols not found: [ _ZNSt8ios_base11__noreplaceE ]
Failed to GetVariableOffset:Symbols not found: [ _ZNSt8ios_base9noreplaceE ]
At time +2.0022096s server received 101 bytes from 10.1.1.1 port 49153


At time +3.0022096s server sent 101 bytes from 10.1.1.1 port 49153


At time +3.0022096s server received 101 bytes from 10.1.1.1 port 49153


At time +4.0022096s server sent 101 bytes from 10.1.1.1 port 49153


At time +4.0022096s server received 101 bytes from 10.1.1.1 port 49153


At time +5.0022096s server sent 101 bytes from 10.1.1.1 port 49153


... (6 outputs left out)

At time +8.0022096s server received 101 bytes from 10.1.1.1 port 49153


At time +9.0022096s server sent 101 bytes from 10.1.1.1 port 49153


At time +9.0022096s server received 101 bytes from 10.1.1.1 port 49153


At time +10.0022096s server sent 101 bytes from 10.1.1.1 port 49153


As we can se printed above, our server application works. Due to how the bindings handles output, the messages may be out of order, but as long as the execution time of the examples worked, everything should be working just fine.

We can also see our Pcap to double check that.

In [5]:
def print_packets_with_tshark(packet_i):
    import pyshark
    cap = pyshark.FileCapture('application-0-0.pcap')
    i = 0
    for packet in cap:
        if packet_i == i:
            print(packet)
            break
        i += 1
    cap.close()
    
# Jupyter shenanigans with Pyshark asyncio code
import concurrent.futures
def exec_async(func, *args, **kwargs):
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
        future = executor.submit(func, *args, **kwargs)
    return future.result()
result = exec_async(print_packets_with_tshark, packet_i=1)

Packet (Length: 131)
Layer PPP
:	Protocol: Internet Protocol version 4 (0x0021)
Layer IP
:	0100 .... = Version: 4
	.... 0101 = Header Length: 20 bytes (5)
	Differentiated Services Field: 0x00 (DSCP: CS0, ECN: Not-ECT)
	0000 00.. = Differentiated Services Codepoint: Default (0)
	.... ..00 = Explicit Congestion Notification: Not ECN-Capable Transport (0)
	Total Length: 129
	Identification: 0x0001 (1)
	000. .... = Flags: 0x0
	0... .... = Reserved bit: Not set
	.0.. .... = Don't fragment: Not set
	..0. .... = More fragments: Not set
	...0 0000 0000 0000 = Fragment Offset: 0
	Time to Live: 64
	Protocol: UDP (17)
	Header Checksum: 0x0000 [validation disabled]
	Header checksum status: Unverified
	Source Address: 10.1.1.1
	Destination Address: 10.1.1.2
	Stream index: 0
Layer UDP
:	Source Port: 49153
	Destination Port: 1234
	Length: 109
	Checksum: 0x0000 [zero-value ignored]
	Checksum Status: Not present
	Stream index: 0
	Stream Packet Number: 2
	Timestamps
	Time since first frame: 1 second
	Ti

We can see above a packet from the EchoClient, which is an IPv4 packet with UDP transport protocol. 

Source address refers to the Node 0, while the target address refers to the Node 1. 

The payload has 101 bytes, as configured in `echoClient.SetAttribute("PacketSize", ns.core.UintegerValue(101))`.

In [6]:
result = exec_async(print_packets_with_tshark, packet_i=2)

Packet (Length: 131)
Layer PPP
:	Protocol: Internet Protocol version 4 (0x0021)
Layer IP
:	0100 .... = Version: 4
	.... 0101 = Header Length: 20 bytes (5)
	Differentiated Services Field: 0x00 (DSCP: CS0, ECN: Not-ECT)
	0000 00.. = Differentiated Services Codepoint: Default (0)
	.... ..00 = Explicit Congestion Notification: Not ECN-Capable Transport (0)
	Total Length: 129
	Identification: 0x0000 (0)
	000. .... = Flags: 0x0
	0... .... = Reserved bit: Not set
	.0.. .... = Don't fragment: Not set
	..0. .... = More fragments: Not set
	...0 0000 0000 0000 = Fragment Offset: 0
	Time to Live: 64
	Protocol: UDP (17)
	Header Checksum: 0x0000 [validation disabled]
	Header checksum status: Unverified
	Source Address: 10.1.1.2
	Destination Address: 10.1.1.1
	Stream index: 0
Layer UDP
:	Source Port: 1234
	Destination Port: 49153
	Length: 109
	Checksum: 0x0000 [zero-value ignored]
	Checksum Status: Not present
	Stream index: 0
	Stream Packet Number: 3
	Timestamps
	Time since first frame: 1.004419000 

We can see above a packet from the EchoServer. 

Same everything, but with a delay of 1 second (`Time since first frame: 1.004419000 seconds`), since our EchoServer sends the message back to the EchoClient a second later.

Source address refers to the Node 1, while the target address refers to the Node 0.

## Audio streaming application using a lossy channel

Even though we know channels are subjected to noise and interference. However, we also have plenty of features built into our network interface (e.g. link negotiation, link adaptation and HARQ, which automatically retransmits corrupted code blocks) and other layers to ensure data gets correctly transmitted to both ends of the communication.

Let's first show how applications behave when no such features are in place.

To do this in an easy-to-understand way, lets use raw media files (wav audio files) as our payloads, since they're more resilient to data loss and decoding corrupted files just results and different outputs for a given input.

In [7]:
import sys
from hashlib import sha256 as sha

ns.cppjit.cppdef("""
    #ifndef StreamingAppSendBlock
    #define StreamingAppSendBlock
    namespace ns3
    {
        EventImpl* pythonMakeEventSendBlock(void (*f)(Ptr<Socket>, Address&), Ptr<Socket> socket, Address address)

        {
            return MakeEvent(f, socket, address);
        }
    }
    #endif
""")
samples_per_block = 200

class StreamingServer(ns.Application):
    LOGGING = False
    STREAMING_PORT = 2345  # Application default listening port
    socketToInstanceDict = {}

    def __init__(self, node: ns.Node, port=STREAMING_PORT, wav_file=None):
        if not wav_file:
            raise Exception("No wav file was specified")
        import os
        if not os.path.exists(wav_file):
            raise Exception("File does not exist", wav_file)
        super().__init__()  # Calls ns.Application constructor
        self.__python_owns__ = False  # Allow C++ to destroy this object when Simulator::Destroy gets called
        self.port = port  # Server listening port
        # Create an UDP socket
        self.m_socket = ns.Socket.CreateSocket(node,
                                               ns.TypeId.LookupByName("ns3::UdpSocketFactory"))
        # Bind that socket to a specific port and IP addresses to listen
        self.m_socket.Bind(ns.InetSocketAddress(ns.Ipv4Address.GetAny(), self.port).ConvertTo())

        # Creates the callback to handle packet reception using a static function of the StreamingServerClass
        # (This is a workaround for the application to work from Python)
        self.m_socket.SetRecvCallback(ns.make_rx_callback(StreamingServer._Receive))

        # Registers the socket as a dictionary key and associate it to this application instance
        StreamingServer.socketToInstanceDict[self.m_socket] = self

        # Load the wav file
        import wave
        wav_file = wave.open(wav_file, "rb")

        # Read blocks of 200 samples
        self.blocks_to_play = []
        data = None
        while data != b"":
            data = wav_file.readframes(samples_per_block)
            self.blocks_to_play.append(data)

        self.blocks_to_play = self.blocks_to_play[2500:7500]

    def __del__(self):
        # Remove the instance entry on the class dictionary when the object is destroyed
        del StreamingServer.socketToInstanceDict[self.m_socket]

    def SendBlock(self, address: ns.Address):
        # Check if list of blocks to transmit ended
        if len(self.blocks_to_play) == 0:
            return

        # Get a block
        block_to_transmit = self.blocks_to_play[0]
        self.blocks_to_play.pop(0)
        #print("sending ", sha(block_to_transmit).hexdigest())
        # Encode bytestream into a ns-3 packet
        packet_with_stream_block = ns.Packet(bytearray(block_to_transmit), len(block_to_transmit))
        packet_with_stream_block.__python_owns__ = False

        self.Send(packet_with_stream_block, address)

        # Re-schedule event until the blocks list run out
        event = ns.pythonMakeEventSendBlock(self._SendBlock, self.m_socket, address)
        ns.Simulator.Schedule(ns.MilliSeconds(60), event)

    def Send(self, packet: ns.Packet, address: ns.Address) -> None:
        # Function that sends the packet to a target address.
        # Assemble a new Packet containing the streamed block
        # Send the Packet to the address
        self.m_socket.SendTo(packet, 0, address)
        if StreamingServer.LOGGING:
            inetAddress = ns.InetSocketAddress.ConvertFrom(address)
            print("At time +{s}s server sent {b} bytes from {ip} port {port}"
                  .format(s=ns.Simulator.Now().GetSeconds(),
                          b=packet.GetSize(),
                          ip=inetAddress.GetIpv4(),
                          port=inetAddress.GetPort()),
                  file=sys.stderr,
                  flush=True)
            if ns.Simulator.Now().GetSeconds() > 5:
                StreamingServer.LOGGING = False

    def Receive(self):
        # Function that receives a packet and the address of the sender
        address = ns.Address()
        packet = self.m_socket.RecvFrom(address)
        if StreamingServer.LOGGING:
            inetAddress = ns.InetSocketAddress.ConvertFrom(address)
            print("At time +{s}s server received {b} bytes from {ip} port {port}"
                  .format(s=ns.Simulator.Now().GetSeconds(),
                          b=packet.__deref__().GetSize(),
                          ip=inetAddress.GetIpv4(),
                          port=inetAddress.GetPort()),
                  file=sys.stderr,
                  flush=True)
            if ns.Simulator.Now().GetSeconds() > 5:
                StreamingServer.LOGGING = False

        # Whoever sends a packet to server will start receiving the stream
        # Start streaming blocks after 1 second
        event = ns.pythonMakeEventSendBlock(self._SendBlock, self.m_socket, address)
        ns.Simulator.Schedule(ns.MilliSeconds(60), event)

    @staticmethod
    def _SendBlock(socket: ns.Socket, address: ns.Address):
        # Static function that identifies the StreamingServer instance that is
        # supposed to send a packet and calls it to send
        instance = StreamingServer.socketToInstanceDict[socket]
        instance.SendBlock(address)

    @staticmethod
    def _Send(socket: ns.Socket, packet: ns.Packet, address: ns.Address):
        # Static function that identifies the StreamingServer instance that is
        # supposed to send a packet and calls it to send
        instance = StreamingServer.socketToInstanceDict[socket]
        instance.Send(packet, address)
        pass

    @staticmethod
    def _Receive(socket: ns.Socket) -> None:
        # Static function that identifies the StreamingServer instance that is
        # supposed to receive a packet and calls it to receive
        instance = StreamingServer.socketToInstanceDict[socket]
        instance.Receive()
        pass


class StreamingClient(ns.Application):
    LOGGING = False
    socketToInstanceDict = {}

    def __init__(self, node: ns.Node, serverAddress: ns.Ipv4Address, port=StreamingServer.STREAMING_PORT,
                 wav_file=None):
        if not wav_file:
            raise Exception("No wav file was specified")
        import os
        if not os.path.exists(wav_file):
            raise Exception("File does not exist", wav_file)
        super().__init__()  # Calls ns.Application constructor
        self.__python_owns__ = False  # Allow C++ to destroy this object when Simulator::Destroy gets called
        self.port = port  # Server listening port
        # Create an UDP socket
        self.m_socket = ns.Socket.CreateSocket(node,
                                               ns.TypeId.LookupByName("ns3::UdpSocketFactory"))

        # Creates the callback to handle packet reception using a static function of the StreamingClient class
        # (This is a workaround for the application to work from Python)
        self.m_socket.SetRecvCallback(ns.make_rx_callback(StreamingClient._Receive))

        # Registers the socket as a dictionary key and associate it to this application instance
        StreamingClient.socketToInstanceDict[self.m_socket] = self

        # Schedule the Send function to send the packet back to its sender after 1 second
        packet = ns.Packet(0)
        packet.__python_owns__ = False
        address = ns.InetSocketAddress(serverAddress, port).ConvertTo()
        event = ns.pythonMakeEventSend(StreamingClient._Send, self.m_socket, packet, address)
        ns.Simulator.Schedule(ns.Seconds(1), event)

        import wave
        self.output_audio = wave.open("audio_out.wav", "wb")
        with wave.open(wav_file, "rb") as f:
            self.output_audio.setsampwidth(f.getsampwidth())
            self.output_audio.setframerate(f.getframerate())
            self.output_audio.setnchannels(f.getnchannels())

    def __del__(self):
        # Remove the instance entry on the class dictionary when the object is destroyed
        del StreamingClient.socketToInstanceDict[self.m_socket]
        self.output_audio.close()

    def Receive(self):
        # Function that receives a packet and the address of the sender
        address = ns.Address()
        packet = self.m_socket.RecvFrom(address)
        if StreamingClient.LOGGING:
            inetAddress = ns.InetSocketAddress.ConvertFrom(address)
            print("At time +{s}s client received {b} bytes from {ip} port {port}"
                  .format(s=ns.Simulator.Now().GetSeconds(),
                          b=packet.__deref__().GetSize(),
                          ip=inetAddress.GetIpv4(),
                          port=inetAddress.GetPort()),
                  file=sys.stderr,
                  flush=True)
            if ns.Simulator.Now().GetSeconds() > 5:
                StreamingClient.LOGGING = False

        # Extract bytestream from Packet and play
        from ctypes import c_uint8

        size = packet.__deref__().GetSize()
        contents_bytes_array = c_uint8*size
        contents = contents_bytes_array()
        packet.__deref__().CopyData(contents, size)
        contents = bytes(contents)
        #print("received ", sha(contents).hexdigest())

        self.output_audio.writeframes(contents)

    @staticmethod
    def _Receive(socket: ns.Socket) -> None:
        # Static function that identifies the StreamingClient instance that is
        # supposed to receive a packet and calls it to receive
        instance = StreamingClient.socketToInstanceDict[socket]
        instance.Receive()
        pass

    def Send(self, packet: ns.Packet, address: ns.Address) -> None:
        # Function that sends the packet to a target address.
        # Assemble a new Packet containing the streamed block
        # Send the Packet to the address
        self.m_socket.SendTo(packet, 0, address)
        if StreamingClient.LOGGING:
            inetAddress = ns.InetSocketAddress.ConvertFrom(address)
            print("At time +{s}s client sent {b} bytes from {ip} port {port}"
                  .format(s=ns.Simulator.Now().GetSeconds(),
                          b=packet.__deref__().GetSize(),
                          ip=inetAddress.GetIpv4(),
                          port=inetAddress.GetPort()),
                  file=sys.stderr,
                  flush=True)
            if ns.Simulator.Now().GetSeconds() > 5:
                StreamingClient.LOGGING = False

    @staticmethod
    def _Send(socket: ns.Socket, packet: ns.Packet, address: ns.Address):
        # Static function that identifies the StreamingClient instance that is
        # supposed to send a packet and calls it to send
        instance = StreamingClient.socketToInstanceDict[socket]
        instance.Send(packet, address)
        pass

Now that we have our applications ready, we can setup our simulation and show the audios for you to listen.

In [8]:
from IPython.display import Audio
from IPython.display import display

wav_file = "./200_the_application_layer/richard-michael-walker_titan.wav"
display(Audio(url=wav_file, embed=False, autoplay=False))

def run_application(lossModel=False, byteErrorRate=0):
    nodes = ns.NodeContainer()
    nodes.Create(2)

    pointToPoint = ns.PointToPointHelper()
    pointToPoint.SetDeviceAttribute("DataRate", ns.StringValue("5Mbps"))
    pointToPoint.SetChannelAttribute("Delay", ns.StringValue("2ms"))

    devices = pointToPoint.Install(nodes)

    if lossModel:
        for i in range(devices.GetN()):
            em = ns.CreateObject[ns.RateErrorModel]()
            em.__deref__().SetAttribute("ErrorRate", ns.DoubleValue(byteErrorRate))
            devices.Get(i).__deref__().SetAttribute("ReceiveErrorModel", ns.PointerValue(em))

    stack = ns.InternetStackHelper()
    stack.Install(nodes)

    address = ns.Ipv4AddressHelper()
    address.SetBase(ns.Ipv4Address("10.1.1.0"),
                    ns.Ipv4Mask("255.255.255.0"))

    interfaces = address.Assign(devices)

    streamServer = StreamingServer(nodes.Get(1), wav_file=wav_file)
    nodes.Get(1).AddApplication(streamServer)
    serverApps = ns.ApplicationContainer()
    serverApps.Add(streamServer)
    serverApps.Start(ns.Seconds(1.0))
    serverApps.Stop(ns.Seconds(10.0))

    address = interfaces.GetAddress(1)

    streamClient = StreamingClient(nodes.Get(0), address, StreamingServer.STREAMING_PORT, wav_file=wav_file)
    clientApps = ns.ApplicationContainer()
    clientApps.Add(streamClient)
    clientApps.Start(ns.Seconds(2.0))
    clientApps.Stop(ns.Seconds(10.0))

    ns.Simulator.Run()
    ns.Simulator.Destroy()

    # Add widget with modified audio
    display(Audio("audio_out.wav", autoplay=False))

In [9]:
run_application()

In [10]:
run_application(lossModel=True, byteErrorRate=0.00001)

In [11]:
run_application(lossModel=True, byteErrorRate=0.0001)

In [12]:
run_application(lossModel=True, byteErrorRate=0.001)

As we can see from the examples above, our data gets more and more distorted as the error rate of our link increases. This is why reliable transfer protocols such as TCP were developed for, detect and recover from packet losses and some corruption.

## Video streaming application in a wireless channel

Instead of using a simplified lossy channel like previously, now we go do the real deal. A more realistic LTE model, with mobility, transmitting an uncompressed video stream.

First, we start by importing our required libraries

In [13]:
from ctypes import c_uint8
import cv2
import ffmpegio
import math
import numpy as np
import pickle

Then, we define our video streaming application. It is a very simple UDP streaming application, where the server starts the stream as soon as it receives the first message from the client, and send one video frame for each received packet. On the client side, we start by sending an empty package and wait for the frame. To ensure hiccups and lost packets don't fully stop the stream, we set a keep-alive event to send a message every two seconds.

In [14]:
ns.cppjit.cppdef(
    """
    #ifndef StreamingAppSendBlock
    #define StreamingAppSendBlock
    namespace ns3
    {
        EventImpl* pythonMakeEventSendBlock(void (*f)(Ptr<Socket>, Address&), Ptr<Socket> socket, Address address)

        {
            return MakeEvent(f, socket, address);
        }
    }
    #endif
"""
)

video = "Welcome_to_Brasilia.mp4"
width = ffmpegio.probe.video_streams_basic(video)[0]["width"]
height = ffmpegio.probe.video_streams_basic(video)[0]["height"]


class VideoStreamingServer(ns.Application):
    STREAMING_PORT = 2345  # Application default listening port
    socketToInstanceDict = {}

    def __init__(self, node: ns.Node, port=STREAMING_PORT):
        super().__init__()  # Calls ns.Application constructor
        self.__python_owns__ = False  # Allow C++ to destroy this object when Simulator::Destroy gets called
        self.port = port  # Server listening port
        # Create an UDP socket
        self.m_socket = ns.Socket.CreateSocket(
            node, ns.TypeId.LookupByName("ns3::UdpSocketFactory")
        )
        # Bind that socket to a specific port and IP addresses to listen
        self.m_socket.Bind(
            ns.InetSocketAddress(ns.Ipv4Address.GetAny(), self.port).ConvertTo()
        )

        # Creates the callback to handle packet reception using a static function of the VideoStreamingServerClass
        # (This is a workaround for the application to work from Python)
        self.m_socket.SetRecvCallback(ns.make_rx_callback(VideoStreamingServer._Receive))

        # Registers the socket as a dictionary key and associate it to this application instance
        VideoStreamingServer.socketToInstanceDict[self.m_socket] = self

        self.framerate = int(ffmpegio.probe.video_streams_basic(video)[0]["frame_rate"])
        self.video = cv2.VideoCapture(video)

    def __del__(self):
        # Remove the instance entry on the class dictionary when the object is destroyed
        del VideoStreamingServer.socketToInstanceDict[self.m_socket]
        cv2.destroyAllWindows()

    def SendBlock(self, address: ns.Address):
        ret, frame = self.video.read()
        if not ret:
            return
        # cv2.imshow("Transmitter", frame)
        # cv2.waitKey(1)

        if ret:
            # convert to byte array
            buffer = frame.tobytes()
            # get size of the frame
            buffer_size = len(buffer)

            num_of_packs = 1
            max_length = 1024
            if buffer_size > max_length:
                num_of_packs = math.ceil(buffer_size / max_length)

            frame_info = {"packs": num_of_packs}

            # Encode bytestream into a ns-3 packet
            block_to_transmit = bytearray(pickle.dumps(frame_info))
            packet_with_stream_block = ns.Packet(
                block_to_transmit, len(block_to_transmit)
            )
            packet_with_stream_block.__python_owns__ = False
            self.Send(packet_with_stream_block, address)

            left = 0
            right = max_length

            for i in range(num_of_packs):
                # truncate data to send
                data = buffer[left:right]
                left = right
                right += max_length

                # send the frames accordingly
                packet_with_stream_block = ns.Packet(bytearray(data), len(data))
                packet_with_stream_block.__python_owns__ = False
                self.Send(packet_with_stream_block, address)

    def Send(self, packet: ns.Packet, address: ns.Address) -> None:
        # Function that sends the packet to a target address.
        # Assemble a new Packet containing the streamed block
        # Send the Packet to the address
        self.m_socket.SendTo(packet, 0, address)

    def Receive(self):
        # Function that receives a packet and the address of the sender
        address = ns.Address()
        packet = self.m_socket.RecvFrom(address)
        event = ns.pythonMakeEventSendBlock(self._SendBlock, self.m_socket, address)
        ns.Simulator.Schedule(ns.MilliSeconds(500 / self.framerate), event)

    @staticmethod
    def _SendBlock(socket: ns.Socket, address: ns.Address):
        # Static function that identifies the VideoStreamingServer instance that is
        # supposed to send a packet and calls it to send
        instance = VideoStreamingServer.socketToInstanceDict[socket]
        instance.SendBlock(address)

    @staticmethod
    def _Send(socket: ns.Socket, packet: ns.Packet, address: ns.Address):
        # Static function that identifies the VideoStreamingServer instance that is
        # supposed to send a packet and calls it to send
        instance = VideoStreamingServer.socketToInstanceDict[socket]
        instance.Send(packet, address)
        pass

    @staticmethod
    def _Receive(socket: ns.Socket) -> None:
        # Static function that identifies the VideoStreamingServer instance that is
        # supposed to receive a packet and calls it to receive
        instance = VideoStreamingServer.socketToInstanceDict[socket]
        instance.Receive()
        pass


class VideoStreamingClient(ns.Application):

    socketToInstanceDict = {}

    def __init__(
        self,
        node: ns.Node,
        serverAddress: ns.Ipv4Address,
        port=VideoStreamingServer.STREAMING_PORT,
    ):
        super().__init__()  # Calls ns.Application constructor
        self.__python_owns__ = False  # Allow C++ to destroy this object when Simulator::Destroy gets called
        self.port = port  # Server listening port
        # Create an UDP socket
        self.m_socket = ns.Socket.CreateSocket(
            node, ns.TypeId.LookupByName("ns3::UdpSocketFactory")
        )

        # Creates the callback to handle packet reception using a static function of the VideoStreamingClient class
        # (This is a workaround for the application to work from Python)
        self.m_socket.SetRecvCallback(ns.make_rx_callback(VideoStreamingClient._Receive))

        # Registers the socket as a dictionary key and associate it to this application instance
        VideoStreamingClient.socketToInstanceDict[self.m_socket] = self

        # Schedule the Send function to send the packet back to its sender after 1 second
        packet = ns.Packet(0)
        packet.__python_owns__ = False
        address = ns.InetSocketAddress(serverAddress, port).ConvertTo()
        event = ns.pythonMakeEventSend(
            VideoStreamingClient._Send, self.m_socket, packet, address
        )
        ns.Simulator.Schedule(ns.Seconds(2), event)

        self.data = bytearray()
        self.num_of_packets = -1
        self.framerate = int(ffmpegio.probe.video_streams_basic(video)[0]["frame_rate"])
        width = int(ffmpegio.probe.video_streams_basic(video)[0]["width"])
        height = int(ffmpegio.probe.video_streams_basic(video)[0]["height"])
        self.video_out = cv2.VideoWriter(
            "video_out.avi",
            cv2.VideoWriter.fourcc("m", "p", "4", "v"),
            self.framerate,
            (width, height),
        )
        self.keepalive = None

    def __del__(self):
        # Remove the instance entry on the class dictionary when the object is destroyed
        del VideoStreamingClient.socketToInstanceDict[self.m_socket]
        self.video_out.release()

    def Receive(self):
        # Function that receives a packet and the address of the sender
        address = ns.Address()
        packet = self.m_socket.RecvFrom(address)
        # Extract bytestream from Packet and play
        size = packet.__deref__().GetSize()
        contents_bytes_array = c_uint8 * size
        contents = contents_bytes_array()
        packet.__deref__().CopyData(contents, size)
        contents = bytes(contents)
        self.data += contents

        # Decode payload
        if len(contents) < 100:
            try:
                frame_info = pickle.loads(contents)
                if frame_info:
                    nums_of_packs = frame_info["packs"]
                    self.num_of_packets = nums_of_packs
                    self.data = bytearray()
            except pickle.UnpicklingError:
                pass
        else:
            self.num_of_packets -= 1

            if self.num_of_packets == 0:
                frame = np.frombuffer(self.data, dtype=np.uint8)
                try:
                    frame = frame.reshape(height, width, 3)
                    self.video_out.write(frame)
                    # cv2.imshow("Receiver", frame)
                    cv2.waitKey(1)
                except ValueError:
                    #print("Dropped frame")
                    pass

                # Transmit null packet to get a new frame
                VideoStreamingClient._SendNullPacket(self.m_socket, address)

    @staticmethod
    def _SendNullPacket(socket: ns.Socket, address: ns.Address) -> None:
        # Retrieve instance
        instance = VideoStreamingClient.socketToInstanceDict[socket]
        if instance.keepalive:
            instance.keepalive.Cancel()
            instance.keepalive = None
        # create null packet to request next frame
        packet = ns.Packet(0)
        packet.__python_owns__ = False
        # Zero out buffer
        instance.data = bytearray()
        instance.Send(packet, address)
        # Schedule keepalive
        instance.keepalive = ns.pythonMakeEventSendBlock(
            VideoStreamingClient._SendNullPacket, socket, address
        )
        ns.Simulator.Schedule(ns.Seconds(2), instance.keepalive)

    @staticmethod
    def _Receive(socket: ns.Socket) -> None:
        # Static function that identifies the VideoStreamingClient instance that is
        # supposed to receive a packet and calls it to receive
        instance = VideoStreamingClient.socketToInstanceDict[socket]
        instance.Receive()
        pass

    def Send(self, packet: ns.Packet, address: ns.Address) -> None:
        # Function that sends the packet to a target address.
        # Assemble a new Packet containing the streamed block
        # Send the Packet to the address
        self.m_socket.SendTo(packet, 0, address)

    @staticmethod
    def _Send(socket: ns.Socket, packet: ns.Packet, address: ns.Address):
        # Static function that identifies the VideoStreamingClient instance that is
        # supposed to send a packet and calls it to send
        instance = VideoStreamingClient.socketToInstanceDict[socket]
        instance.Send(packet, address)
        pass

Now we can define our simulation. Very simple setup that can have either a static UE at 100 m from the base station, or moving back and forth (up to 1.1 km away, and back to 100 m). Error models are enabled and transmission power is very low 10 dBm on purpose. Channel bandwidth is of 20 MHz (100 RBs).

Here is the video we are going to stream, as a reference.

In [15]:
from IPython.display import Video
from IPython.display import display
display(Video("Welcome_to_Brasilia.mp4", width=720, height=480)) # The source video is 720p

In [16]:
def run_application(static_ue=True):
    nodes = ns.NodeContainer()
    nodes.Create(2)

    ns.Config.SetDefault(
        "ns3::LteRlcUm::MaxTxBufferSize", ns.UintegerValue(100 * 1024 * 1024)
    )

    rbs = 100
    ns.Config.SetDefault("ns3::ComponentCarrier::DlBandwidth", ns.UintegerValue(rbs))
    ns.Config.SetDefault("ns3::ComponentCarrier::UlBandwidth", ns.UintegerValue(rbs))
    ns.Config.SetDefault("ns3::LteEnbNetDevice::DlBandwidth", ns.UintegerValue(rbs))
    ns.Config.SetDefault("ns3::LteEnbNetDevice::UlBandwidth", ns.UintegerValue(rbs))
    ns.Config.SetDefault("ns3::LteEnbPhy::TxPower", ns.DoubleValue(10))
    ns.Config.SetDefault("ns3::LteUePhy::TxPower", ns.DoubleValue(10))
    ns.Config.SetDefault(
        "ns3::LteSpectrumPhy::CtrlErrorModelEnabled", ns.BooleanValue(True)
    )
    ns.Config.SetDefault(
        "ns3::LteSpectrumPhy::DataErrorModelEnabled", ns.BooleanValue(True)
    )
    lteHelper = ns.CreateObject[ns.LteHelper]()
    epcHelper = ns.CreateObject[ns.PointToPointEpcHelper]()
    lteHelper.SetEpcHelper(epcHelper)
    pgw = epcHelper.GetPgwNode()

    # Create a single RemoteHost
    remoteHostContainer = ns.NodeContainer()
    remoteHostContainer.Create(1)
    remoteHost = remoteHostContainer.Get(0)
    internet = ns.InternetStackHelper()
    internet.Install(remoteHostContainer)

    # Create the Internet
    p2ph = ns.PointToPointHelper()
    p2ph.SetDeviceAttribute("DataRate", ns.DataRateValue(ns.DataRate("100Gb/s")))
    p2ph.SetDeviceAttribute("Mtu", ns.UintegerValue(1500))
    p2ph.SetChannelAttribute("Delay", ns.TimeValue(ns.Seconds(0.010)))
    internetDevices = p2ph.Install(pgw, remoteHost)
    ipv4h = ns.Ipv4AddressHelper()
    ipv4h.SetBase("1.0.0.0", "255.0.0.0")
    internetIpIfaces = ipv4h.Assign(internetDevices)
    # interface 0 is localhost, 1 is the p2p device
    remoteHostAddr = internetIpIfaces.GetAddress(1)

    ipv4RoutingHelper = ns.Ipv4StaticRoutingHelper()
    remoteHostStaticRouting = ipv4RoutingHelper.GetStaticRouting(
        remoteHost.GetObject[ns.Ipv4]()
    )
    remoteHostStaticRouting.AddNetworkRouteTo(
        ns.Ipv4Address("7.0.0.0"), ns.Ipv4Mask("255.0.0.0"), 1
    )

    ueNodes = ns.NodeContainer()
    enbNodes = ns.NodeContainer()
    enbNodes.Add(nodes.Get(0))
    ueNodes.Add(nodes.Get(1))

    # Install Mobility Model
    gnbPositionAlloc = ns.CreateObject[ns.ListPositionAllocator]()
    gnbPositionAlloc.Add(ns.Vector(100, 0, 0))
    mobility = ns.MobilityHelper()
    mobility.SetMobilityModel("ns3::ConstantPositionMobilityModel")
    mobility.SetPositionAllocator(gnbPositionAlloc)
    mobility.Install(enbNodes)

    uePositionAlloc = ns.CreateObject[ns.ListPositionAllocator]()
    uePositionAlloc.Add(ns.Vector(200, 200, 0))
    if static_ue:
        mobility.SetPositionAllocator(uePositionAlloc)
        mobility.Install(ueNodes)
    else:
        mobility.SetMobilityModel("ns3::WaypointMobilityModel")
        mobility.SetPositionAllocator(uePositionAlloc)
        mobility.Install(ueNodes)

        pos = ns.Vector(200, 200, 0)
        i = 0
        for dir in [1, -1, 1, -1, 1, -1, 1, -1]:
            for _ in range(10):
                time = ns.Seconds(i * 10)
                nextPos = ns.Vector(pos.x + dir * 100, pos.y, 0)
                wpt = ns.Waypoint(time, nextPos)
                ueNodes.Get(0).GetObject[ns.WaypointMobilityModel]().AddWaypoint(wpt)
                pos = nextPos
                i += 1

    # Install LTE Devices to the nodes
    enbLteDevs = lteHelper.InstallEnbDevice(enbNodes)
    ueLteDevs = lteHelper.InstallUeDevice(ueNodes)

    # Install the IP stack on the UEs
    internet.Install(ueNodes)
    epcHelper.AssignUeIpv4Address(ns.NetDeviceContainer(ueLteDevs))
    # Set the default gateway for the UE
    ueStaticRouting = ipv4RoutingHelper.GetStaticRouting(
        ueNodes.Get(0).GetObject[ns.Ipv4]()
    )
    ueStaticRouting.SetDefaultRoute(epcHelper.GetUeDefaultGatewayAddress(), 1)
    lteHelper.Attach(ueLteDevs.Get(0), enbLteDevs.Get(0))

    streamServer = VideoStreamingServer(remoteHost)
    remoteHost.AddApplication(streamServer)
    serverApps = ns.ApplicationContainer()
    serverApps.Add(streamServer)
    serverApps.Start(ns.Seconds(1))

    streamClient = VideoStreamingClient(
        nodes.Get(1), remoteHostAddr, VideoStreamingServer.STREAMING_PORT
    )
    clientApps = ns.ApplicationContainer()
    clientApps.Add(streamClient)
    clientApps.Start(ns.Seconds(2))

    ns.Simulator.Stop(ns.Seconds(800))
    ns.Simulator.Run()
    ns.Simulator.Destroy()
    
    # Add widget with modified video
    import subprocess
    video_out = "video_{}.mp4".format('static' if static_ue else 'dynamic')
    subprocess.check_output(["ffmpeg", "-y", "-i", "video_out.avi", video_out])
    display(Video(video_out, width=720, height=480))


run_application()

Failed to GetVariableOffset:Symbols not found: [ _ZN3ns310int64x64_t14implementationE ]


ffmpeg version 8.0.1-3ubuntu2 Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 15 (Ubuntu 15.2.0-13ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu2 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-pocketsphinx --disable-libcaca --disable-libmfx --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml

[avi @ 0x64acc1f68f80] Packet corrupt (stream = 0, dts = 693).
[in#0/avi @ 0x64acc1f68cc0] corrupt input packet in stream 0
[mpeg4 @ 0x64acc1f92140] ac-tex damaged at 69 44
[mpeg4 @ 0x64acc1f92140] Error at MB: 5393
[mpeg4 @ 0x64acc1f92140] concealing 2860 DC, 2860 AC, 2860 MV errors in P frame
[vist#0:0/mpeg4 @ 0x64acc1ff5580] [dec:mpeg4 @ 0x64acc1f6eec0] corrupt decoded frame


[out#0/mp4 @ 0x64acc1ff5740] video:14981KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.048035%
frame=  694 fps=112 q=-1.0 Lsize=   14989KiB time=00:00:27.68 bitrate=4435.9kbits/s speed=4.48x elapsed=0:00:06.17    
[libx264 @ 0x64acc1f6cf40] frame I:7     Avg QP:17.87  size:183693
[libx264 @ 0x64acc1f6cf40] frame P:410   Avg QP:21.97  size: 30381
[libx264 @ 0x64acc1f6cf40] frame B:277   Avg QP:26.18  size:  5770
[libx264 @ 0x64acc1f6cf40] consecutive B-frames: 41.4% 13.5%  8.2% 36.9%
[libx264 @ 0x64acc1f6cf40] mb I  I16..4: 22.9% 60.0% 17.1%
[libx264 @ 0x64acc1f6cf40] mb P  I16..4:  1.5%  4.4%  0.1%  P16..4: 40.9% 12.7%  8.5%  0.0%  0.0%    skip:31.9%
[libx264 @ 0x64acc1f6cf40] mb B  I16..4:  0.8%  1.4%  0.0%  B16..8: 42.3%  1.2%  0.1%  direct: 0.8%  skip:53.3%  L0:46.5% L1:46.7% BI: 6.8%
[libx264 @ 0x64acc1f6cf40] 8x8 transform intra:69.3% inter:67.6%
[libx264 @ 0x64acc1f6cf40] coded y,uvDC,uvAC intra: 42.4% 45.6% 5.7% inter: 16.8% 10.8% 0.1%
[li

And now with mobility, going out-of-range and back to range.

In [17]:
run_application(static_ue=False)

ffmpeg version 8.0.1-3ubuntu2 Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 15 (Ubuntu 15.2.0-13ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu2 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-pocketsphinx --disable-libcaca --disable-libmfx --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml

[avi @ 0x5cafb9081f80] Packet corrupt (stream = 0, dts = 105).
[in#0/avi @ 0x5cafb9081cc0] corrupt input packet in stream 0
[mpeg4 @ 0x5cafb90b6400] concealing 1154 DC, 1154 AC, 1154 MV errors in P frame


[vist#0:0/mpeg4 @ 0x5cafb910e580] [dec:mpeg4 @ 0x5cafb9087ec0] corrupt decoded frame


[out#0/mp4 @ 0x5cafb910e740] video:2239KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.083599%
frame=  106 fps= 77 q=-1.0 Lsize=    2241KiB time=00:00:04.16 bitrate=4413.5kbits/s speed=3.02x elapsed=0:00:01.37    
[libx264 @ 0x5cafb9085f40] frame I:4     Avg QP:16.99  size:148530
[libx264 @ 0x5cafb9085f40] frame P:56    Avg QP:22.50  size: 25242
[libx264 @ 0x5cafb9085f40] frame B:46    Avg QP:24.62  size:  6191
[libx264 @ 0x5cafb9085f40] consecutive B-frames: 36.8% 13.2%  8.5% 41.5%
[libx264 @ 0x5cafb9085f40] mb I  I16..4: 29.2% 63.7%  7.0%
[libx264 @ 0x5cafb9085f40] mb P  I16..4:  2.0%  6.8%  0.1%  P16..4: 38.1% 10.4%  6.8%  0.0%  0.0%    skip:35.7%
[libx264 @ 0x5cafb9085f40] mb B  I16..4:  2.9%  3.2%  0.0%  B16..8: 36.3%  1.2%  0.1%  direct: 2.8%  skip:53.4%  L0:48.5% L1:46.2% BI: 5.3%
[libx264 @ 0x5cafb9085f40] 8x8 transform intra:66.2% inter:73.1%
[libx264 @ 0x5cafb9085f40] coded y,uvDC,uvAC intra: 39.0% 36.4% 4.9% inter: 12.7% 9.2% 0.1%
[libx